In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
from xgboost import XGBClassifier
from model_setup_colorado import X as Xco, y as yco, df as dfco

here = Path.cwd()
repo_root = next(p for p in [here, *here.parents] if (p / "data").exists())

Rows: 343 | Schools: 22
Share labeled 1: 0.417


Cell 2: build the same features for every program

In [2]:
nat = pd.read_csv(repo_root / "data" / "victoria_datasets" / "clean_dataset.csv",
                  dtype={"COUNTY_FIPS": str}, low_memory=False)
print("programs:", len(nat), "| with school info:", nat["HAS_INSTITUTION"].sum())
nat = nat[nat["HAS_INSTITUTION"]].copy()   # the rest have no school features to score

nat["CIP_FAMILY"] = nat["CIPCODE"] // 100
nat["LOCALE_GROUP"] = nat["LOCALE"] // 10
for c in ["IPEDSCOUNT1", "UGDS", "B01003_001E"]:
    nat[c + "_LOG"] = np.log1p(nat[c])
ctrl = pd.get_dummies(nat["CONTROL_inst"].astype("string"), prefix="CTRL", dtype=float)

base_cols = [c for c in Xco.columns if not c.startswith("CTRL_")]
Xnat = pd.concat([nat[base_cols], ctrl], axis=1)
Xnat = Xnat.reindex(columns=Xco.columns, fill_value=0.0).astype(float)
print(Xnat.shape, "| same columns as Colorado:", list(Xnat.columns) == list(Xco.columns))

programs: 43465 | with school info: 41357
(41357, 19) | same columns as Colorado: True


Cell 3: the outcome part

In [3]:
model = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                      subsample=0.8, random_state=42, eval_metric="logloss")
model.fit(Xco, yco)
nat["OUTCOME_PROB"] = model.predict_proba(Xnat)[:, 1]

# Only score subjects with at least 5 Colorado programs
counts = dfco["CIP_FAMILY"].value_counts()
nat["SUBJECT_COVERED"] = nat["CIP_FAMILY"].isin(counts[counts >= 5].index)
nat.loc[~nat["SUBJECT_COVERED"], "OUTCOME_PROB"] = np.nan

# Flag programs the model was trained on (their scores are not out-of-sample)
key = nat["UNITID"].astype("Int64").astype(str) + "_" + nat["CIPCODE"].astype("Int64").astype(str)
train_key = dfco["UNITID"].astype("Int64").astype(str) + "_" + dfco["CIPCODE"].astype("Int64").astype(str)
nat["IN_COLORADO_TRAINING"] = key.isin(set(train_key))

Cell 4: the demand part and checks

In [4]:
nat["DEMAND_SCORE"] = (nat["TOT_EMP"].rank(pct=True) + nat["LOC_QUOTIENT"].rank(pct=True)) / 2 * 100
nat["SCHOOL_COMPLETION"] = nat["C150_L4"]

has_o = nat["OUTCOME_PROB"].notna()
has_d = nat["DEMAND_SCORE"].notna()
print("programs:", len(nat))
print("with an outcome score:", has_o.sum(), "| with a demand score:", has_d.sum(), "| both:", (has_o & has_d).sum())
print("in Colorado training set:", nat["IN_COLORADO_TRAINING"].sum())
print("\noutcome score distribution:")
print(nat["OUTCOME_PROB"].describe().round(2).to_string())
print("\nrank correlation, outcome vs demand:",
      round(nat[["OUTCOME_PROB", "DEMAND_SCORE"]].corr(method="spearman").iloc[0, 1], 2))

display(nat[has_o].groupby("CIP_FAMILY")
        .agg(programs=("OUTCOME_PROB", "size"), avg_outcome=("OUTCOME_PROB", "mean"))
        .sort_values("programs", ascending=False).head(12).round(2))

print("\nduplicate school+program rows:", nat.duplicated(["UNITID", "CIPCODE"]).sum())
nat[nat.duplicated(["UNITID", "CIPCODE"], keep=False)].sort_values(["UNITID", "CIPCODE"])[
    ["UNITID", "CIPCODE", "CIPDESC", "IPEDSCOUNT1", "EARN_MDN_1YR_CLEAN"]].head(8)

programs: 41357
with an outcome score: 34408 | with a demand score: 24316 | both: 20245
in Colorado training set: 281

outcome score distribution:
count    34408.00
mean         0.45
std          0.25
min          0.02
25%          0.23
50%          0.43
75%          0.66
max          0.98

rank correlation, outcome vs demand: -0.1


,programs,avg_outcome
CIP_FAMILY,,
51,8236,0.30
52,5662,0.54
11,3216,0.79
15,3057,0.60
47,2457,0.34
12,2189,0.11
46,1623,0.62
43,1581,0.73
50,1451,0.26



duplicate school+program rows: 0


,UNITID,CIPCODE,CIPDESC,IPEDSCOUNT1,EARN_MDN_1YR_CLEAN


Check

In [5]:
sc = nat[nat["OUTCOME_PROB"].notna()]
tot = sc["OUTCOME_PROB"].var()
within = (sc["OUTCOME_PROB"] - sc.groupby("CIP_FAMILY")["OUTCOME_PROB"].transform("mean")).var()
print("share of score variation explained by subject alone:", round(1 - within / tot, 2))

share of score variation explained by subject alone: 0.61


Save

In [6]:
out_pct = nat["OUTCOME_PROB"].rank(pct=True) * 100
nat["COMBINED"] = pd.concat([out_pct, nat["DEMAND_SCORE"]], axis=1).mean(axis=1)   # averages whatever parts exist
nat.loc[nat["OUTCOME_PROB"].isna(), "COMBINED"] = np.nan

nat["TIER"] = pd.qcut(nat["COMBINED"], 3, labels=["Low", "Medium", "High"]).astype("string").fillna("Not scored")
nat["TIER_BASIS"] = np.where(nat["OUTCOME_PROB"].isna(), "not scored",
                     np.where(nat["DEMAND_SCORE"].isna(), "outcome only", "outcome + demand"))

print(nat.groupby(["TIER_BASIS", "TIER"]).size().unstack(fill_value=0))

cols = ["UNITID", "INSTNM_inst", "STABBR", "CBSA Title", "CIPCODE", "CIPDESC", "CIP_FAMILY",
        "OUTCOME_PROB", "DEMAND_SCORE", "SCHOOL_COMPLETION", "COMBINED", "TIER", "TIER_BASIS",
        "IN_COLORADO_TRAINING"]
out = nat[cols].round(3)
out.to_csv(repo_root / "data" / "victoria_datasets" / "readiness_scores.csv", index=False)
print(out.shape)

TIER              High   Low  Medium  Not scored
TIER_BASIS                                      
not scored           0     0       0        6949
outcome + demand  6035  5795    8415           0
outcome only      5434  5675    3054           0
(41357, 14)
